# data import

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from os.path  import join

import random
import itertools
import math



import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
import xgboost as xgb
from xgboost import XGBClassifier
import optuna

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss

In [ ]:
data_path = "../../data"

# The Basics ------------------------------------------------------------------------
# Men
MTeams = pd.read_csv(join(data_path, 'MTeams.csv'))
MSeasons = pd.read_csv(join(data_path, 'MSeasons.csv'))
MNCAATourneySeeds = pd.read_csv(join(data_path, 'MNCAATourneySeeds.csv'))
MRegularSeasonCompactResults = pd.read_csv(join(data_path, 'MRegularSeasonCompactResults.csv'))
MNCAATourneyCompactResults = pd.read_csv(join(data_path, 'MNCAATourneyCompactResults.csv'))
# Women
WTeams = pd.read_csv(join(data_path, 'WTeams.csv'))
WSeasons = pd.read_csv(join(data_path, 'WSeasons.csv'))
WNCAATourneySeeds = pd.read_csv(join(data_path, 'WNCAATourneySeeds.csv'))
WRegularSeasonCompactResults = pd.read_csv(join(data_path, 'WRegularSeasonCompactResults.csv'))
WNCAATourneyCompactResults = pd.read_csv(join(data_path, 'WNCAATourneyCompactResults.csv'))
# Other
SampleSubmissionStage1 = pd.read_csv(join(data_path, 'SampleSubmissionStage1.csv'))
SampleSubmissionStage2 = pd.read_csv(join(data_path, 'SampleSubmissionStage2.csv'))
SeedBenchmarkStage1 = pd.read_csv(join(data_path, 'SeedBenchmarkStage1.csv'))

# Team Box Scores ------------------------------------------------------------------------
# Men
MRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'MRegularSeasonDetailedResults.csv'))
MNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'MNCAATourneyDetailedResults.csv'))
# Women
WRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'WRegularSeasonDetailedResults.csv'))
WNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'WNCAATourneyDetailedResults.csv'))

# Geography ------------------------------------------------------------------------
# All
Cities = pd.read_csv(join(data_path, 'Cities.csv'))
Conferences = pd.read_csv(join(data_path, 'Conferences.csv'))
# Men
MGameCities = pd.read_csv(join(data_path, 'MGameCities.csv'))
# Women
WGameCities = pd.read_csv(join(data_path, 'WGameCities.csv'))

# Public Rankings ------------------------------------------------------------------------
# Men
MMasseyOrdinals = pd.read_csv(join(data_path, 'MMasseyOrdinals.csv')) # men only

# Supplements ------------------------------------------------------------------------
# Men
MTeamCoaches = pd.read_csv(join(data_path, 'MTeamCoaches.csv')) # men only
MTeamConferences = pd.read_csv(join(data_path, 'MTeamConferences.csv'))
MConferenceTourneyGames = pd.read_csv(join(data_path, 'MConferenceTourneyGames.csv'))
MSecondaryTourneyTeams = pd.read_csv(join(data_path, 'MSecondaryTourneyTeams.csv'))
MSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'MSecondaryTourneyCompactResults.csv'))
MTeamSpellings = pd.read_csv(join(data_path, "MTeamSpellings.csv"), encoding='cp1252')
MNCAATourneySlots = pd.read_csv(join(data_path, 'MNCAATourneySlots.csv'))
MNCAATourneySeedRoundSlots = pd.read_csv(join(data_path, 'MNCAATourneySeedRoundSlots.csv')) # men only
# Women
WTeamConferences = pd.read_csv(join(data_path, 'WTeamConferences.csv'))
WConferenceTourneyGames = pd.read_csv(join(data_path, 'WConferenceTourneyGames.csv'))
WSecondaryTourneyTeams = pd.read_csv(join(data_path, 'WSecondaryTourneyTeams.csv'))
WSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'WSecondaryTourneyCompactResults.csv'))
WTeamSpellings = pd.read_csv(join(data_path, 'WTeamSpellings.csv'), encoding='cp1252')
WNCAATourneySlots = pd.read_csv(join(data_path, 'WNCAATourneySlots.csv'))

In [ ]:
games = MRegularSeasonDetailedResults[['Season', 'DayNum', 'WTeamID', 'LTeamID']].copy().drop_duplicates()

# Rolling stats computation

In [ ]:
def compute_rolling_stats(match_df, Team_id, Season, DayNum, n_matches=5):
    stat_columns = [
        'Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 
        'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF'
    ]
    
    # Filter matches where the team was involved
    match_df = match_df[(match_df['WTeamID'] == Team_id) | (match_df['LTeamID'] == Team_id)]
    
    # Sort by Season and DayNum
    match_df = match_df.sort_values(['Season', 'DayNum'])
    
    # Filter only matches before the given Season and DayNum
    match_df = match_df[(match_df['Season'] < Season) | ((match_df['Season'] == Season) & (match_df['DayNum'] < DayNum))]
    
    # Take the last `n_matches`
    last_n_matches = match_df.tail(n_matches).copy()  # Copy to avoid modifying the original dataframe

    for col in stat_columns:
        last_n_matches[col] = last_n_matches.apply(
            lambda row: row[f'W{col}'] if row['WTeamID'] == Team_id else row[f'L{col}'], axis=1
        )

    # Compute rolling averages for the selected matches
    rolling_stats = last_n_matches[stat_columns].mean()

    return last_n_matches, rolling_stats


# Target computation

In [ ]:
# data_path = '../../data'

# # The Basics ------------------------------------------------------------------------
# # Men
# MTeams = pd.read_csv(join(data_path, 'MTeams.csv'))
# MSeasons = pd.read_csv(join(data_path, 'MSeasons.csv'))
# MNCAATourneySeeds = pd.read_csv(join(data_path, 'MNCAATourneySeeds.csv'))
# MRegularSeasonCompactResults = pd.read_csv(join(data_path, 'MRegularSeasonCompactResults.csv'))
# MNCAATourneyCompactResults = pd.read_csv(join(data_path, 'MNCAATourneyCompactResults.csv'))
# # Women
# WTeams = pd.read_csv(join(data_path, 'WTeams.csv'))
# WSeasons = pd.read_csv(join(data_path, 'WSeasons.csv'))
# WNCAATourneySeeds = pd.read_csv(join(data_path, 'WNCAATourneySeeds.csv'))
# WRegularSeasonCompactResults = pd.read_csv(join(data_path, 'WRegularSeasonCompactResults.csv'))
# WNCAATourneyCompactResults = pd.read_csv(join(data_path, 'WNCAATourneyCompactResults.csv'))
# # Other
# SampleSubmissionStage1 = pd.read_csv(join(data_path, 'SampleSubmissionStage1.csv'))
# SampleSubmissionStage2 = pd.read_csv(join(data_path, 'SampleSubmissionStage2.csv'))
# SeedBenchmarkStage1 = pd.read_csv(join(data_path, 'SeedBenchmarkStage1.csv'))

# # Team Box Scores ------------------------------------------------------------------------
# # Men
# MRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'MRegularSeasonDetailedResults.csv'))
# MNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'MNCAATourneyDetailedResults.csv'))
# # Women
# WRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'WRegularSeasonDetailedResults.csv'))
# WNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'WNCAATourneyDetailedResults.csv'))

# # Geography ------------------------------------------------------------------------
# # All
# Cities = pd.read_csv(join(data_path, 'Cities.csv'))
# Conferences = pd.read_csv(join(data_path, 'Conferences.csv'))
# # Men
# MGameCities = pd.read_csv(join(data_path, 'MGameCities.csv'))
# # Women
# WGameCities = pd.read_csv(join(data_path, 'WGameCities.csv'))

# # Public Rankings ------------------------------------------------------------------------
# # Men
# MMasseyOrdinals = pd.read_csv(join(data_path, 'MMasseyOrdinals.csv')) # men only

# # Supplements ------------------------------------------------------------------------
# # Men
# MTeamCoaches = pd.read_csv(join(data_path, 'MTeamCoaches.csv')) # men only
# MTeamConferences = pd.read_csv(join(data_path, 'MTeamConferences.csv'))
# MConferenceTourneyGames = pd.read_csv(join(data_path, 'MConferenceTourneyGames.csv'))
# MSecondaryTourneyTeams = pd.read_csv(join(data_path, 'MSecondaryTourneyTeams.csv'))
# MSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'MSecondaryTourneyCompactResults.csv'))
# MTeamSpellings = pd.read_csv(join(data_path, "MTeamSpellings.csv"), encoding='cp1252')
# MNCAATourneySlots = pd.read_csv(join(data_path, 'MNCAATourneySlots.csv'))
# MNCAATourneySeedRoundSlots = pd.read_csv(join(data_path, 'MNCAATourneySeedRoundSlots.csv')) # men only
# # Women
# WTeamConferences = pd.read_csv(join(data_path, 'WTeamConferences.csv'))
# WConferenceTourneyGames = pd.read_csv(join(data_path, 'WConferenceTourneyGames.csv'))
# WSecondaryTourneyTeams = pd.read_csv(join(data_path, 'WSecondaryTourneyTeams.csv'))
# WSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'WSecondaryTourneyCompactResults.csv'))
# WTeamSpellings = pd.read_csv(join(data_path, 'WTeamSpellings.csv'), encoding='cp1252')
# WNCAATourneySlots = pd.read_csv(join(data_path, 'WNCAATourneySlots.csv'))

# V2

In [ ]:
season_year = 2024

## Helper Functions

In [ ]:
def create_features_and_target(games_df, n_matches=5):
    # List of stat columns (without the W/L prefixes)
    stats = ['Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 
             'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']

    # --- Build a long DataFrame with one row per team-game ---
    # For winning teams (label 1)
    w_df = games_df[['Season', 'DayNum', 'WTeamID'] + [f'W{stat}' for stat in stats]].copy()
    w_df.rename(columns={'WTeamID': 'TeamID'}, inplace=True)
    for stat in stats:
        w_df[stat] = w_df[f'W{stat}']
    w_df['Won'] = 1
    w_df = w_df[['Season', 'DayNum', 'TeamID'] + stats + ['Won']]
    
    # For losing teams (label 0)
    l_df = games_df[['Season', 'DayNum', 'LTeamID'] + [f'L{stat}' for stat in stats]].copy()
    l_df.rename(columns={'LTeamID': 'TeamID'}, inplace=True)
    for stat in stats:
        l_df[stat] = l_df[f'L{stat}']
    l_df['Won'] = 0
    l_df = l_df[['Season', 'DayNum', 'TeamID'] + stats + ['Won']]
    
    # Combine winners and losers into one DataFrame
    combined = pd.concat([w_df, l_df], ignore_index=True)
    combined.sort_values(['TeamID', 'Season', 'DayNum'], inplace=True)
    combined.reset_index(drop=True, inplace=True)
    
    # --- Compute rolling averages for the stats ---
    # For each team, compute rolling averages over the previous n_matches
    rolling = combined.groupby('TeamID')[stats].apply(
        lambda group: group.shift(1).rolling(window=n_matches, min_periods=1).mean()
    ).reset_index(drop=True)
    rolling = rolling.add_suffix('_rolling')
    
    # Combine the rolling averages with the original DataFrame
    combined_stats = pd.concat([combined, rolling], axis=1)
    
    # --- Separate the features and target ---
    # Now include Season, DayNum, and TeamID in the features
    feature_cols = ["Season", "DayNum", "TeamID"] + [f"{stat}_rolling" for stat in stats]
    X = combined_stats[feature_cols].copy()
    y = combined_stats['Won'].copy()
    
    return X, y, combined_stats

def get_teamid_in_tournament(tournament_results, year):
    tournament_results = tournament_results[tournament_results['Season'] == year]
    teams = set(tournament_results['WTeamID']).union(set(tournament_results['LTeamID']))
    return teams

def filter_stats(combined_stats, teams_in_tourney):
    return combined_stats[combined_stats['TeamID'].isin(teams_in_tourney)]

def create_X_and_y(combined_stats,  teams_in_tournament, regular_results):
    filtered_stats = filter_stats(combined_stats, teams_in_tourney)
    games_with_teams_in_tournament = regular_results[
        regular_results['WTeamID'].isin(teams_in_tourney) | regular_results['LTeamID'].isin(teams_in_tourney)
    ]
    games_with_teams_in_tournament = games_with_teams_in_tournament[['Season', 'DayNum', 'WTeamID', 'LTeamID']]
    
    # Create a DataFrame where Team1 is the winner and Team2 is the loser
    games_with_teams_12 = games_with_teams_in_tournament.copy()
    games_with_teams_12 = games_with_teams_12.rename(columns={'WTeamID': 'Team1', 'LTeamID': 'Team2'})
    games_with_teams_12['Won'] = 1  # Team1 won

    # Create a DataFrame where Team1 is the loser and Team2 is the winner
    games_with_teams_21 = games_with_teams_in_tournament.copy()
    games_with_teams_21 = games_with_teams_21.rename(columns={'LTeamID': 'Team1', 'WTeamID': 'Team2'})
    games_with_teams_21['Won'] = 0  # Team1 lost
    
    # Combine both DataFrames
    games_combined = pd.concat([games_with_teams_12, games_with_teams_21], ignore_index=True)

    # print(len(games_with_teams_in_tournament))
    # print(len(games_combined))
    # print(games_combined.columns)
    
    return games_combined


def join_pairings_with_team_stats(pairings_l, combined_st, join_on_day = True, season = None):
    # Define the join keys
    if join_on_day:
        join_keys = ['Season', 'DayNum', 'TeamID']
        # Create a copy of combined_stats and add the prefix "team_1_" to non-key columns
        combined_stats_team1 = combined_st.copy()
        cols_to_prefix = [col for col in combined_stats_team1.columns if col not in join_keys]
        combined_stats_team1 = combined_stats_team1.rename(
            columns={col: f"team_1_{col}" for col in cols_to_prefix}
        )

        # Now merge pairings with the renamed combined_stats on Season, DayNum, and Team1 == TeamID
        pairings_combined_with_stats = pd.merge(
            pairings_l,
            combined_stats_team1,
            how='left',
            left_on=['Season', 'DayNum', 'Team1'],
            right_on=['Season', 'DayNum', 'TeamID']
        )

        # Create a copy of combined_stats and add the prefix "team_2_" to non-key columns
        combined_stats_team2 = combined_st.copy()
        combined_stats_team2 = combined_stats_team2.rename(
            columns={col: f"team_2_{col}" for col in cols_to_prefix}
        )

        # Now merge pairings_combined_with_stats with the renamed combined_stats on Season, DayNum, and Team2 == TeamID
        pairings_combined_with_stats = pd.merge(
            pairings_combined_with_stats,
            combined_stats_team2,
            how='left',
            left_on=['Season', 'DayNum', 'Team2'],
            right_on=['Season', 'DayNum', 'TeamID']
        )

        # Remove NaN values

        # pairings_combined_with_stats.dropna(inplace=True)
        # return pairings_combined_with_stats
    else:
        join_keys = ['Season', 'DayNum', 'TeamID']
        # print(combined_st.columns, combined_st.shape)
        # Filter for the given season and keep only the last record per team (by DayNum)
        combined_st = combined_st[combined_st['Season'] == season]
        combined_st = combined_st.sort_values('DayNum').drop_duplicates(subset='TeamID', keep='last')

        combined_stats_team1 = combined_st.copy()
        cols_to_prefix = [col for col in combined_stats_team1.columns if col not in ['Season', 'DayNum', 'TeamID']]
        combined_stats_team1 = combined_stats_team1.rename(
            columns={col: f"team_1_{col}" for col in cols_to_prefix}
        )        
        # print("Combinde", combined_stats_team1.columns, combined_stats_team1.shape)

        pairings_combined_with_stats = pd.merge(
            pairings_l,
            combined_stats_team1,
            how='left',
            left_on=['Season', 'Team1'],
            right_on=['Season', 'TeamID']
        )

        combined_stats_team2 = combined_st.copy()
        cols_to_prefix = [col for col in combined_stats_team2.columns if col not in ['Season', 'DayNum', 'TeamID']]
        combined_stats_team2 = combined_stats_team2.rename(
            columns={col: f"team_2_{col}" for col in cols_to_prefix}
        )        

        pairings_combined_with_stats = pd.merge(
            pairings_combined_with_stats,
            combined_stats_team2,
            how='left',
            left_on=['Season',  'Team2'],
            right_on=['Season', 'TeamID']
        )
        pairings_combined_with_stats.drop(columns=['DayNum_y', 'TeamID_y'], inplace=True)

    


    pairings_combined_with_stats.dropna(inplace=True)
    return pairings_combined_with_stats

def generate_X_y_from_pairings(pairings):

    desired_columns = [
    'team_1_Score_rolling', 'team_1_FGM_rolling', 'team_1_FGA_rolling',
    'team_1_FGM3_rolling', 'team_1_FGA3_rolling', 'team_1_FTM_rolling',
    'team_1_FTA_rolling', 'team_1_OR_rolling', 'team_1_DR_rolling',
    'team_1_Ast_rolling', 'team_1_TO_rolling', 'team_1_Stl_rolling',
    'team_1_Blk_rolling', 'team_1_PF_rolling', 'team_2_Score_rolling',
    'team_2_FGM_rolling', 'team_2_FGA_rolling', 'team_2_FGM3_rolling',
    'team_2_FGA3_rolling', 'team_2_FTM_rolling', 'team_2_FTA_rolling',
    'team_2_OR_rolling', 'team_2_DR_rolling', 'team_2_Ast_rolling',
    'team_2_TO_rolling', 'team_2_Stl_rolling', 'team_2_Blk_rolling',
    'team_2_PF_rolling'
    ]


    # columns_to_drop = ['Season', 'DayNum', 'Team1', 'Team2', 'Won', 'team_1_Score', 'team_2_Score', 'team_1_Won', 'team_2_Won', 'TeamID_x', 'team_1_FGM', 'team_1_FGA', 'team_1_FGM3', 'team_1_FGA3',
    #    'team_1_FTM', 'team_1_FTA', 'team_1_OR', 'team_1_DR', 'team_1_Ast',
    #    'team_1_TO', 'team_1_Stl', 'team_1_Blk', 'team_1_PF',
    #      'TeamID_y', 
    #      'team_2_FGM',
    #    'team_2_FGA', 'team_2_FGM3', 'team_2_FGA3', 'team_2_FTM', 'team_2_FTA',
    #    'team_2_OR', 'team_2_DR', 'team_2_Ast', 'team_2_TO', 'team_2_Stl',
    #    'team_2_Blk', 'team_2_PF']

    return pairings[desired_columns].copy(), pairings['Won']

## Manual prediction

In [ ]:
teams_in_tourney = get_teamid_in_tournament(MNCAATourneyCompactResults, 2024)
len(teams_in_tourney)

In [ ]:
X, y, combined_stats = create_features_and_target(MRegularSeasonDetailedResults, n_matches=5)

In [ ]:
print(combined_stats.columns)
combined_stats

In [ ]:
pairings = create_X_and_y(combined_stats, teams_in_tourney, MRegularSeasonDetailedResults)

In [ ]:
pairings_combined_with_stats = join_pairings_with_team_stats(pairings, combined_stats)


In [ ]:
nan_counts = pairings_combined_with_stats.isna().sum()
print(nan_counts)
print(pairings_combined_with_stats.shape)

In [ ]:
pairings_combined_with_stats.columns

In [ ]:
print(pairings[(pairings['Season'] == 2024) & (pairings['DayNum'] == 132)])
print(MRegularSeasonDetailedResults[(MRegularSeasonDetailedResults['Season'] == 2024) & (MRegularSeasonDetailedResults['DayNum'] == 132)])

In [ ]:
X, y = generate_X_y_from_pairings(pairings_combined_with_stats)

In [ ]:
X.columns

# Tournament data preperation

In [ ]:
MRegularSeasonCompactResults.columns

In [ ]:
MNCAATourneyCompactResults.columns

In [ ]:
tournament_pairings = create_X_and_y(combined_stats, teams_in_tourney, MNCAATourneyCompactResults)
tournament_pairings = tournament_pairings[tournament_pairings['Season']==season_year]
tournament_pairings

In [ ]:
regular_season_and_tournament_combined = pd.concat(
    [MRegularSeasonDetailedResults, MNCAATourneyDetailedResults],
    ignore_index=True
).copy()

join_pairings_with_team_stats(tournament_pairings, combined_stats, join_on_day=False, season=season_year)


In [ ]:
pairings_combined_with_stats_tournament = join_pairings_with_team_stats(pairings, combined_stats)
nan_counts = pairings_combined_with_stats_tournament.isna().sum()
print(nan_counts)
print(pairings_combined_with_stats_tournament.shape)


X_tournament, y_tournament = generate_X_y_from_pairings(pairings_combined_with_stats_tournament)

# Models

## Basic LR

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegressionCV(
        cv=5,                # 5-fold cross-validation
        solver='lbfgs',
        max_iter=2000,       # Increase max_iter if needed
        scoring='neg_log_loss',  # Optimize log-loss for probability calibration
        refit=True
    ))
])

# Train the model using the pipeline
pipeline.fit(X, y)


In [ ]:
X.columns

In [ ]:
y_tournament

In [ ]:
y_prob = pipeline.predict_proba(X_tournament)[:, 1]

score = brier_score_loss(y_tournament, y_prob)
print("Brier score:", score)

In [ ]:
df_results = pd.DataFrame({
    'y_prob': y_prob,
    'y_tournament': y_tournament
})
df_results['difference'] = df_results['y_prob'] - df_results['y_tournament']
df_results.head()


In [ ]:
import numpy as np

import matplotlib.pyplot as plt

# Get the coefficients from the logistic regression model
coefficients = pipeline.named_steps['clf'].coef_[0]

# Get the feature names
feature_names = X.columns

# Create a DataFrame to hold the feature names and their corresponding coefficients
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

# Sort the DataFrame by the absolute value of the coefficients
importance_df['AbsCoefficient'] = np.abs(importance_df['Coefficient'])
importance_df = importance_df.sort_values(by='AbsCoefficient', ascending=False)

# Plot the feature importances
plt.figure(figsize=(10, 8))
plt.barh(importance_df['Feature'], importance_df['Coefficient'])
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# pd.concat([y_prob, y_tournament], axis=1)

In [ ]:
import pickle
from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold

In [ ]:
pickle.dump(X, open("X.pkl", "wb"))
pickle.dump(y, open("y.pkl", "wb"))

# Finding the best # of matches to look back

## Helpers 

In [ ]:
MRegularSeasonDetailedResults[MRegularSeasonDetailedResults['Season'] == 2024].shape
MSecondaryTourneyCompactResults[MSecondaryTourneyCompactResults['Season'] == 2024].shape    

In [87]:
def get_X_y_X_tournament_y_tournament(season_year, number_of_matches_back):
    teams_in_tournament = get_teamid_in_tournament(MNCAATourneyCompactResults, season_year)

    # Get all regular season features and target
    X, y, combined_stats = create_features_and_target(MRegularSeasonDetailedResults, n_matches=number_of_matches_back)
    
    # Filter to only include rows for the desired season
    idx = X['Season'] == season_year
    X = X[idx].copy()
    y = y[idx].copy()
    combined_stats = combined_stats[combined_stats['Season'] == season_year].copy()
    
    print("X,y after season filter:", X.shape, y.shape)

    # Create pairings from the regular season games (for teams in tourney)
    pairings = create_X_and_y(combined_stats, teams_in_tournament, MRegularSeasonDetailedResults)
    pairings_combined_with_stats = join_pairings_with_team_stats(pairings, combined_stats)
    # Optionally check for NaNs:
    # nan_counts = pairings_combined_with_stats.isna().sum()
    
    X, y = generate_X_y_from_pairings(pairings_combined_with_stats)
    print("X,y after pairing join:", X.shape, y.shape)

    # Tournament data processing
    tournament_pairings = create_X_and_y(combined_stats, teams_in_tournament, MNCAATourneyCompactResults)
    tournament_pairings = tournament_pairings[tournament_pairings['Season'] == season_year]
    
    # Optionally combine detailed results (if needed)
    regular_season_and_tournament_combined = pd.concat(
        [MRegularSeasonDetailedResults, MNCAATourneyDetailedResults],
        ignore_index=True
    ).copy()

    pairings_combined_with_stats_tournament = join_pairings_with_team_stats(
        tournament_pairings, combined_stats, join_on_day=False, season=season_year
    )
    pairings_combined_with_stats_tournament.dropna(inplace=True)
    # print(pairings_combined_with_stats_tournament.shape)
    
    X_tournament, y_tournament = generate_X_y_from_pairings(pairings_combined_with_stats_tournament)
    print("X_tournament, y_tournament:", X_tournament.shape, y_tournament.shape)

    return X, y, X_tournament, y_tournament


In [90]:
a,b,c,d = get_X_y_X_tournament_y_tournament(2024, 5)

print(a.shape, b.shape, c.shape, d.shape)
print(MRegularSeasonDetailedResults[MRegularSeasonDetailedResults['Season'] == 2024].shape, print(MRegularSeasonDetailedResults[MRegularSeasonDetailedResults['Season'] == 2024].shape), MNCAATourneyCompactResults[MNCAATourneyCompactResults['Season'] == 2024].shape)

X,y after season filter: (11214, 17) (11214,)
X,y after pairing join: (3804, 28) (3804,)
X_tournament, y_tournament: (134, 28) (134,)
(3804, 28) (3804,) (134, 28) (134,)
(5607, 34)
(5607, 34) None (67, 8)


In [ ]:
def create_plot(model):
    # Get the coefficients from the logistic regression model
    coefficients = model.named_steps['clf'].coef_[0]

    # Get the feature names
    feature_names = X.columns

    # Create a DataFrame to hold the feature names and their corresponding coefficients
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Coefficient': coefficients
    })

    # Sort the DataFrame by the absolute value of the coefficients
    importance_df['AbsCoefficient'] = np.abs(importance_df['Coefficient'])
    importance_df = importance_df.sort_values(by='AbsCoefficient', ascending=False)

    # Plot the feature importances
    plt.figure(figsize=(10, 8))
    plt.barh(importance_df['Feature'], importance_df['Coefficient'])
    plt.xlabel('Coefficient Value')
    plt.ylabel('Feature')
    plt.title('Feature Importance')
    plt.gca().invert_yaxis()
    plt.show()

## LR

In [ ]:
def create_model(season_year, number_of_matches_back):

    teams_in_tourney = get_teamid_in_tournament(MNCAATourneyCompactResults, season_year)

    X, y, combined_stats = create_features_and_target(MRegularSeasonDetailedResults, n_matches=number_of_matches_back)

    pairings = create_X_and_y(combined_stats, teams_in_tourney, MRegularSeasonDetailedResults)
    pairings_combined_with_stats = join_pairings_with_team_stats(pairings, combined_stats)
    nan_counts = pairings_combined_with_stats.isna().sum()

    X, y = generate_X_y_from_pairings(pairings_combined_with_stats)

    tournament_pairings = create_X_and_y(combined_stats, teams_in_tourney, MNCAATourneyCompactResults)
    tournament_pairings = tournament_pairings[tournament_pairings['Season']==season_year]
    tournament_pairings

    regular_season_and_tournament_combined = pd.concat(
        [MRegularSeasonDetailedResults, MNCAATourneyDetailedResults],
        ignore_index=True
    ).copy()

    # join_pairings_with_team_stats(tournament_pairings, combined_stats, join_on_day=False, season=season_year)


    pairings_combined_with_stats_tournament = join_pairings_with_team_stats(tournament_pairings, combined_stats, join_on_day=False, season=season_year)
    nan_counts = pairings_combined_with_stats_tournament.isna().sum()
    # print(nan_counts)
    # print(pairings_combined_with_stats_tournament.shape)


    X_tournament, y_tournament = generate_X_y_from_pairings(pairings_combined_with_stats_tournament)



    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegressionCV(
            cv=5,                # 5-fold cross-validation
            solver='lbfgs',
            max_iter=2000,       # Increase max_iter if needed
            scoring='neg_log_loss',  # Optimize log-loss for probability calibration
            refit=True
        ))
    ])

    # Train the model using the pipeline
    pipeline.fit(X, y)


    y_prob = pipeline.predict_proba(X_tournament)[:, 1]

    score = brier_score_loss(y_tournament, y_prob)
    print("Brier score:", score)


    df_results = pd.DataFrame({
        'y_prob': y_prob,
        'y_tournament': y_tournament
    })
    df_results['difference'] = df_results['y_prob'] - df_results['y_tournament']
    df_results.head()



    

    return pipeline, score, y_tournament, y_prob, X_tournament

## NN

In [ ]:
def create_neural_net(season_year, number_of_matches_back):
    # Get tournament team IDs
    teams_in_tourney = get_teamid_in_tournament(MNCAATourneyCompactResults, season_year)
    
    # Regular season features
    X, y, combined_stats = create_features_and_target(MRegularSeasonDetailedResults, n_matches=number_of_matches_back)
    
    # Create pairings from regular season results and join stats
    pairings = create_X_and_y(combined_stats, teams_in_tourney, MRegularSeasonDetailedResults)
    pairings_combined_with_stats = join_pairings_with_team_stats(pairings, combined_stats)
    # (Optional) Check for NaNs:
    # nan_counts = pairings_combined_with_stats.isna().sum()
    
    X, y = generate_X_y_from_pairings(pairings_combined_with_stats)
    
    # Prepare tournament data
    tournament_pairings = create_X_and_y(combined_stats, teams_in_tourney, MNCAATourneyCompactResults)
    tournament_pairings = tournament_pairings[tournament_pairings['Season'] == season_year]
    
    # Combine regular season and tournament detailed results if needed
    regular_season_and_tournament_combined = pd.concat(
        [MRegularSeasonDetailedResults, MNCAATourneyDetailedResults],
        ignore_index=True
    ).copy()
    
    # Join tournament pairings with team stats (without join_on_day, for season filtering)
    pairings_combined_with_stats_tournament = join_pairings_with_team_stats(tournament_pairings, combined_stats, join_on_day=False, season=season_year)
    # nan_counts = pairings_combined_with_stats_tournament.isna().sum()
    
    X_tournament, y_tournament = generate_X_y_from_pairings(pairings_combined_with_stats_tournament)
    
    # Build a neural network pipeline with scaling and an MLPClassifier
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=(100,), max_iter=2000, random_state=42))
    ])
    
    # Train the model on regular season data
    pipeline.fit(X, y)
    
    # Predict probabilities on tournament data
    y_prob = pipeline.predict_proba(X_tournament)[:, 1]
    score = brier_score_loss(y_tournament, y_prob)
    print("Neural Net Brier score:", score)
    
    # Optional: Create a DataFrame with results for inspection
    df_results = pd.DataFrame({
        'y_prob': y_prob,
        'y_tournament': y_tournament
    })
    df_results['difference'] = df_results['y_prob'] - df_results['y_tournament']
    print(df_results.head())
    
    return pipeline, score, y_tournament, y_prob, X_tournament




## RF

In [ ]:
def create_random_forest(season_year, number_of_matches_back):
    # Get tournament team IDs
    teams_in_tourney = get_teamid_in_tournament(MNCAATourneyCompactResults, season_year)
    
    # Regular season features
    X, y, combined_stats = create_features_and_target(MRegularSeasonDetailedResults, n_matches=number_of_matches_back)
    
    # Create pairings from regular season results and join stats
    pairings = create_X_and_y(combined_stats, teams_in_tourney, MRegularSeasonDetailedResults)
    pairings_combined_with_stats = join_pairings_with_team_stats(pairings, combined_stats)
    # (Optional) Check for NaNs:
    # nan_counts = pairings_combined_with_stats.isna().sum()
    
    X, y = generate_X_y_from_pairings(pairings_combined_with_stats)
    
    # Prepare tournament data
    tournament_pairings = create_X_and_y(combined_stats, teams_in_tourney, MNCAATourneyCompactResults)
    tournament_pairings = tournament_pairings[tournament_pairings['Season'] == season_year]
    
    regular_season_and_tournament_combined = pd.concat(
        [MRegularSeasonDetailedResults, MNCAATourneyDetailedResults],
        ignore_index=True
    ).copy()
    
    pairings_combined_with_stats_tournament = join_pairings_with_team_stats(tournament_pairings, combined_stats, join_on_day=False, season=season_year)
    # nan_counts = pairings_combined_with_stats_tournament.isna().sum()
    
    X_tournament, y_tournament = generate_X_y_from_pairings(pairings_combined_with_stats_tournament)
    
    # Build a random forest pipeline with scaling and a RandomForestClassifier
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
    ])
    
    # Train the model on regular season data
    pipeline.fit(X, y)
    
    # Predict probabilities on tournament data
    y_prob = pipeline.predict_proba(X_tournament)[:, 1]
    score = brier_score_loss(y_tournament, y_prob)
    print("Random Forest Brier score:", score)
    
    # Optional: Create a DataFrame with results for inspection
    df_results = pd.DataFrame({
        'y_prob': y_prob,
        'y_tournament': y_tournament
    })
    df_results['difference'] = df_results['y_prob'] - df_results['y_tournament']
    print(df_results.head())
    
    return pipeline, score, y_tournament, y_prob, X_tournament



## PLOTS

### LR - comparison 

In [ ]:
scores = []
x_vals = []

for i in range(20):
    season_year = 2024
    number_of_matches_back = i + 1
    # Assuming create_model returns: pipeline, score, y_tournament, y_prob
    pipeline, score, y_tournament, y_prob, temp = create_model(season_year, number_of_matches_back)
    scores.append(score)
    x_vals.append(number_of_matches_back)

# Create the plot
plt.figure(figsize=(8, 6))
plt.plot(x_vals, scores, marker='o', linestyle='-')
plt.xlabel("Number of Matches Back")
plt.ylabel("Score")
plt.title("Model Score vs Number of Matches Back")
plt.grid(True)
plt.show()

In [84]:
correct_counts = []
incorrect_counts = []
x_vals = []

for i in range(10):
    season_year = 2024
    number_of_matches_back = i + 1
    # Assume create_model returns: pipeline, score, y_tournament, y_prob
    pipeline, score, y_tournament, y_prob, temp = create_model(season_year, number_of_matches_back)
    
    # Convert probabilities to binary predictions (assuming threshold of 0.5)
    y_pred = (y_prob >= 0.5).astype(int)
    
    # Compute number of correct and incorrect predictions
    correct = (y_pred == y_tournament).sum()
    incorrect = (y_pred != y_tournament).sum()
    
    correct_counts.append(correct)
    incorrect_counts.append(incorrect)
    x_vals.append(number_of_matches_back)

# Convert x_vals to a numpy array for plotting
x_vals = np.array(x_vals)
width = 0.35

# Create grouped bar plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x_vals - width/2, correct_counts, width, label='Correct Predictions')
ax.bar(x_vals + width/2, incorrect_counts, width, label='Incorrect Predictions')

ax.set_xlabel("Number of Matches Back")
ax.set_ylabel("Number of Matches")
ax.set_title("Correct vs. Incorrect Predictions per Iteration")
ax.set_xticks(x_vals)
ax.legend()

plt.show()

KeyboardInterrupt: 

In [ ]:
from joblib import Parallel, delayed
import numpy as np
import matplotlib.pyplot as plt

def run_iteration(i):
    season_year = 2024
    number_of_matches_back = i + 1
    # Assume create_model returns: pipeline, score, y_tournament, y_prob
    pipeline, score, y_tournament, y_prob, temp = create_model(season_year, number_of_matches_back)
    
    # Convert probabilities to binary predictions (using threshold 0.5)
    y_pred = (y_prob >= 0.5).astype(int)
    
    # Compute number of correct and incorrect predictions
    correct = (y_pred == y_tournament).sum()
    incorrect = (y_pred != y_tournament).sum()
    
    return number_of_matches_back, correct, incorrect

# Run iterations in parallel using 16 cores
results = Parallel(n_jobs=16)(delayed(run_iteration)(i) for i in range(64))

# Unpack the results
x_vals, correct_counts, incorrect_counts = zip(*results)
x_vals = np.array(x_vals)
correct_counts = np.array(correct_counts)
incorrect_counts = np.array(incorrect_counts)

# Create the plot
plt.figure(figsize=(10, 6))
width = 0.35
plt.bar(x_vals - width/2, correct_counts, width, label='Correct Predictions')
plt.bar(x_vals + width/2, incorrect_counts, width, label='Incorrect Predictions')
plt.xlabel("Number of Matches Back")
plt.ylabel("Number of Matches")
plt.title("Correct vs. Incorrect Predictions per Iteration")
plt.xticks(x_vals)
plt.legend()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n_iterations = 10
error_matrix = []  # Each row corresponds to an iteration's squared errors per match

for i in range(n_iterations):
    season_year = 2024
    number_of_matches_back = i + 1
    # Assume create_model returns: pipeline, score, y_tournament, y_prob
    pipeline, score, y_tournament, y_prob, temp = create_model(season_year, number_of_matches_back)
    
    # Convert y_tournament and y_prob to numpy arrays (if not already)
    y_tournament = np.array(y_tournament)
    y_prob = np.array(y_prob)
    print(y_prob.shape, "shape")
    
    # Compute squared error for each match
    errors = (y_tournament - y_prob) ** 2
    error_matrix.append(errors)

# Convert list to a 2D numpy array (shape: iterations x number_of_matches)
error_matrix = np.array(error_matrix)

plt.figure(figsize=(10, 6))
# Use imshow to display the matrix; origin='lower' so that iteration 1 is at the bottom
im = plt.imshow(error_matrix, aspect='auto', cmap='Reds', origin='lower')
plt.colorbar(im, label="Squared Error")
plt.xlabel("Match Index")
plt.ylabel("Iteration (Number of Matches Back)")
plt.title("Squared Error per Match vs. Iteration")

# Set x-ticks to show match numbers, and y-ticks to show iteration values
n_matches = error_matrix.shape[1]
plt.xticks(np.arange(n_matches), np.arange(1, n_matches + 1))
plt.yticks(np.arange(n_iterations), np.arange(1, n_iterations + 1))

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n_iterations = 10
error_matrix = []  # Each row corresponds to an iteration's squared errors per match

for i in range(n_iterations):
    season_year = 2024
    number_of_matches_back = i + 1
    # Assume create_model returns: pipeline, score, y_tournament, y_prob, temp
    pipeline, score, y_tournament, y_prob, temp = create_model(season_year, number_of_matches_back)
    
    # Convert y_tournament and y_prob to numpy arrays (if not already)
    y_tournament = np.array(y_tournament)
    y_prob = np.array(y_prob)
    print(y_prob.shape, "shape")
    
    # Round predictions to 0 or 1
    y_pred_rounded = np.round(y_prob)
    
    # Compute squared error using the rounded predictions
    errors = (y_tournament - y_pred_rounded) ** 2
    error_matrix.append(errors)

# Convert list to a 2D numpy array (shape: iterations x number_of_matches)
error_matrix = np.array(error_matrix)

plt.figure(figsize=(10, 6))
# Display the error matrix as a heatmap, with the first iteration at the bottom.
im = plt.imshow(error_matrix, aspect='auto', cmap='Reds', origin='lower')
plt.colorbar(im, label="Squared Error")
plt.xlabel("Match Index")
plt.ylabel("Iteration (Number of Matches Back)")
plt.title("Squared Error per Match vs. Iteration (Rounded Predictions)")

# Set x-ticks and y-ticks to show match numbers and iteration numbers
n_matches = error_matrix.shape[1]
plt.xticks(np.arange(n_matches), np.arange(1, n_matches + 1))
plt.yticks(np.arange(n_iterations), np.arange(1, n_iterations + 1))

plt.show()


### Comparing models

In [ ]:
from joblib import Parallel, delayed
import numpy as np
import matplotlib.pyplot as plt

def run_iteration(i):
    season_year = 2024
    number_of_matches_back = i + 1
    # Assume create_model returns: pipeline, score, y_tournament, y_prob, temp
    pipeline, score, y_tournament, y_prob, temp = create_model(season_year, number_of_matches_back)
    
    # Convert probabilities to binary predictions using a threshold of 0.5
    y_pred = (y_prob >= 0.5).astype(int)
    
    # Compute number of correct and incorrect predictions
    correct = (y_pred == y_tournament).sum()
    incorrect = (y_pred != y_tournament).sum()
    
    return number_of_matches_back, correct, incorrect

# Run iterations in parallel using 16 CPU cores over 64 iterations
results = Parallel(n_jobs=16)(delayed(run_iteration)(i) for i in range(64))

# Unpack results into separate lists
x_vals, correct_counts, incorrect_counts = zip(*results)

# Optional plotting: grouped bar chart of correct vs. incorrect predictions
plt.figure(figsize=(10, 6))
width = 0.4
x_vals = np.array(x_vals)
plt.bar(x_vals - width/2, correct_counts, width, label='Correct Predictions')
plt.bar(x_vals + width/2, incorrect_counts, width, label='Incorrect Predictions')
plt.xlabel("Number of Matches Back")
plt.ylabel("Number of Predictions")
plt.title("Correct vs. Incorrect Predictions per Iteration")
plt.xticks(x_vals)
plt.legend()
plt.grid(True)
plt.show()



# tabpfn

In [ ]:
from tabpfn import TabPFNClassifier

In [ ]:
MRegularSeasonDetailedResults

In [ ]:
X_train, y_train, X_test, y_test = get_X_y_X_tournament_y_tournament(2024, 5)
X_train

In [ ]:
X_test

In [ ]:
MRegularSeasonDetailedResults[MRegularSeasonDetailedResults['Season'] == 2024]

In [ ]:
X_train

In [ ]:
clf = TabPFNClassifier()

In [ ]:

clf.fit(X_train, y_train)

# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
predictions = clf.predict(X_test)

In [ ]:
score = brier_score_loss(y_test, predictions)
print("Neural Net Brier score:", score)